# 02 — RL Agent Training

Trains 4 agents per ticker (PPO and DQN × price-only and price+sentiment):

| Agent | State |
|-------|-------|
| PPO   | price-only |
| PPO   | price + sentiment |
| DQN   | price-only |
| DQN   | price + sentiment |

**Note:** SAC requires a continuous action space — DQN is the correct off-policy algorithm for our discrete (reduce/hold/increase) action space.

**Train period:** Oct 1 – Nov 30, 2025 (43 trading days)
**Test period:** Dec 1 – Dec 31, 2025

Models saved to `models/`.

In [2]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from trading_env import TradingEnv, make_envs

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

MODELS_DIR = PROJECT_ROOT / 'llm/models'
DATA_DIR   = PROJECT_ROOT / 'llm/data'
MODELS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. Load Dataset

In [3]:
daily = pd.read_csv(DATA_DIR / 'daily_features.csv', parse_dates=['date'])

TICKERS = ['NVDA', 'GOOG', 'TSLA']
TRAIN_CUTOFF = '2025-11-30'

print('Dataset shape:', daily.shape)
print('Tickers:', daily['ticker'].unique())
print()

for t in TICKERS:
    sub = daily[daily['ticker'] == t]
    train = sub[sub['date'] <= TRAIN_CUTOFF]
    test  = sub[sub['date'] >  TRAIN_CUTOFF]
    print(f'{t}: {len(train)} train days, {len(test)} test days')

Dataset shape: (192, 33)
Tickers: ['GOOG' 'NVDA' 'TSLA']

NVDA: 42 train days, 22 test days
GOOG: 42 train days, 22 test days
TSLA: 42 train days, 22 test days


## 2. Sanity Check — Environment

In [4]:
# Keep penalties at zero for the main feature ablation.
# Raise these in a separate reward-design experiment.
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.0,
)

# Quick environment checks on NVDA
envs = make_envs(daily, 'NVDA', TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

for env_key, label in [
    ('train_price', 'price-only env'),
    ('train_sentiment_basic', 'basic sentiment env'),
    ('train_sentiment_enhanced', 'enhanced sentiment env'),
]:
    print(f'Checking {label}...')
    check_env(envs[env_key], warn=True)
    print('OK')

print(f"Obs dim (price-only):         {envs['train_price'].observation_space.shape[0]}")
print(f"Obs dim (sentiment_basic):   {envs['train_sentiment_basic'].observation_space.shape[0]}")
print(f"Obs dim (sentiment_enhanced): {envs['train_sentiment_enhanced'].observation_space.shape[0]}")


Checking price-only env...
OK
Checking basic sentiment env...
OK
Checking enhanced sentiment env...
OK
Obs dim (price-only):         10
Obs dim (sentiment_basic):   17
Obs dim (sentiment_enhanced): 33


In [5]:
# Quick rollout to confirm the enhanced env works end-to-end
env = envs['train_sentiment_enhanced']
obs, _ = env.reset()
done = False
total_reward = 0
while not done:
    action = env.action_space.sample()
    obs, reward, done, _, info = env.step(action)
    total_reward += reward

print(f'Random agent episode done. Total reward: {total_reward:.4f}')
print(f'Final portfolio: ${info["portfolio_value"]:.2f}')


Random agent episode done. Total reward: -0.0101
Final portfolio: $9968.02


## 3. Training Configuration

In [8]:
# 30k timesteps is sufficient for our 43-day training window.
# Each episode is ~43 steps, so this is ~700 full episodes per agent.
# Increase to 100k+ if you want longer convergence runs (takes ~5 min/model).
TOTAL_TIMESTEPS = 30_000

# Keep penalties at zero for the main feature ablation.
# Raise these in a separate reward-design experiment.
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.4,
)

PPO_KWARGS = dict(
    policy='MlpPolicy',
    n_steps=43,         # one full episode per rollout
    batch_size=43,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=3e-4,
    verbose=0,
)

DQN_KWARGS = dict(
    policy='MlpPolicy',
    learning_rate=1e-3,
    buffer_size=5_000,
    learning_starts=43,
    batch_size=32,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    exploration_final_eps=0.05,
    exploration_fraction=0.3,
    verbose=0,
)

print(f'Training {TOTAL_TIMESTEPS:,} timesteps per agent.')
print('Environment kwargs:', ENV_KWARGS)
print('Configs set.')


Training 30,000 timesteps per agent.
Environment kwargs: {'reward_mode': 'log_return', 'drawdown_penalty': 0.0, 'volatility_penalty': 0.4}
Configs set.


## 4. Train All Agents

This trains 4 agents × 3 tickers = **12 models** total. At 30k timesteps each model takes ~50s (PPO) or ~30s (DQN).

In [10]:
from tqdm.notebook import tqdm

trained_models = {}  # key: (ticker, algo, state_type)

configs = [
    ('PPO', 'price',               PPO, PPO_KWARGS, 'train_price'),
    ('PPO', 'sentiment_basic',     PPO, PPO_KWARGS, 'train_sentiment_basic'),
    ('PPO', 'sentiment_enhanced',  PPO, PPO_KWARGS, 'train_sentiment_enhanced'),
    ('DQN', 'price',               DQN, DQN_KWARGS, 'train_price'),
    ('DQN', 'sentiment_basic',     DQN, DQN_KWARGS, 'train_sentiment_basic'),
    ('DQN', 'sentiment_enhanced',  DQN, DQN_KWARGS, 'train_sentiment_enhanced'),
]

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

    for algo_name, state_type, AlgoClass, kwargs, env_key in tqdm(configs, desc=ticker):
        env = Monitor(envs[env_key])
        model = AlgoClass(env=env, seed=42, **kwargs)
        model.learn(total_timesteps=TOTAL_TIMESTEPS)

        key = (ticker, algo_name, state_type)
        trained_models[key] = model

        save_path = MODELS_DIR / f'{ticker}_{algo_name}_{state_type}'
        model.save(str(save_path))
        print(f'  Saved {save_path.name}')

print('\nAll models trained.')



=== NVDA ===


NVDA:   0%|          | 0/6 [00:00<?, ?it/s]

UnboundLocalError: local variable 'reward' referenced before assignment

## 5. Quick Training Sanity Check

Run each model on its test set and print final portfolio value.

In [7]:
def evaluate_model(model, env):
    """Run one episode and return history + final portfolio value."""
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
    return env.get_history(), info['portfolio_value']

print(f'Initial cash: $10,000.00\n')
print(f'{"Ticker":<6} {"Algo":<5} {"State":<20} {"Final $":>10}')
print('-' * 48)

test_results = {}
for ticker in TICKERS:
    envs = make_envs(daily, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)
    for algo_name, state_type, AlgoClass, _, env_key in configs:
        test_env_key = env_key.replace('train_', 'test_')
        key = (ticker, algo_name, state_type)
        model = trained_models[key]
        history, final_val = evaluate_model(model, envs[test_env_key])
        test_results[key] = (history, final_val)
        print(f'{ticker:<6} {algo_name:<5} {state_type:<20} ${final_val:>9.2f}')


Initial cash: $10,000.00

Ticker Algo  State                   Final $
------------------------------------------------


NVDA   PPO   price                $  9880.56
NVDA   PPO   sentiment_basic      $ 10121.82
NVDA   PPO   sentiment_enhanced   $ 10087.81
NVDA   DQN   price                $ 10296.39
NVDA   DQN   sentiment_basic      $ 10418.89
NVDA   DQN   sentiment_enhanced   $  9796.91
GOOG   PPO   price                $ 10101.01
GOOG   PPO   sentiment_basic      $ 10080.34
GOOG   PPO   sentiment_enhanced   $  9867.86
GOOG   DQN   price                $  9813.22
GOOG   DQN   sentiment_basic      $ 10016.68


GOOG   DQN   sentiment_enhanced   $  9995.83
TSLA   PPO   price                $ 10544.98
TSLA   PPO   sentiment_basic      $ 10438.33
TSLA   PPO   sentiment_enhanced   $ 10424.22
TSLA   DQN   price                $ 10446.36
TSLA   DQN   sentiment_basic      $  9940.93
TSLA   DQN   sentiment_enhanced   $ 10432.25
